---

MRI analysis 

---

In [1]:
# autoload
%load_ext autoreload
%autoreload 2

# load relevant pgl classes
from pgl import pgl, pglDatabase, pglActions, pglFlywheel, pglAFNI, pglActionsMRI, pglActionsMNE, pglActionsFixes
from pgl.pglExternalTools import pglGLMsingle

---

Load behavioral data

---

In [ ]:
pgl.importSettings()

In [ ]:
newSession = pglActions.loadSession.execute()
if newSession: 
    session = newSession
    session.print()

---

Fix missing volume triggers

---

In [ ]:
fixVolumeTimes = pglActionsFixes.fixVolumeTimes()
newSession = fixVolumeTimes.execute(session=session)
if newSession: 
    session = newSession
    session.print()

In [ ]:
session.display()

In [ ]:
session.print()

---

Load data from flywheel

---

In [ ]:
newSession = pglActions.loadFlywheel().execute(session=session, showScanCounts=False)
if newSession: 
    session = newSession
    session.print()

In [ ]:
session.mri.print(match='nsdmini')

---

Align MRI scans with behavior/experiment session data

---

In [ ]:
session.setScanRunMatches([(4, 0), (5, 1), (6, 2), (10, 3), (11, 4), (12, 5), (17,6), (21, 7), (22, 8), (23, 9)])
#session.setScanRunMatches([(4, 0)])

In [ ]:
session.displayScanRunMapping()

---

Save the raw session to database

---

In [ ]:
session.saveRawSession()

---

Help on commands

---

In [ ]:
# Get a list of all actions within an action module
#pglActionsFixes.help()
#pglActions.help()
#pglActionsMNE.help()
pglActionsMRI.help()

In [ ]:
# Get specific help on a paritcular action
help(pglActionsMRI.mriEstimateMotionAFNI)

---

Database

---

In [ ]:
# Database settings
pglDatabase.settings()

In [2]:
# check status
pglDatabase.status()

=============================== Database status ================================
databaseName = local
databasePath = ~/data/pgl
username     = justin

✅ PostgreSQL configuration is valid.
✅ PostgreSQL is reachable and accepting connections.
✅ Application database and configured schema are accessible.
✅ Schema is up to date: 16 migration(s) verified.


In [ ]:
# display database
pglDatabase.print(experimentName='nsd-mini')
#pglDatabase.print()

---

Load a session

---

In [2]:
# load a specific session
#session = pglDatabase.loadSession(126)
#session = pglDatabase.loadSession(128)
session = pglDatabase.loadSession(136)

(pglDatabase:loadSession) Loading sessionID=136 from local database: ~/data/pgl, username=justin
================================= Session 136 ==================================
  Acquired         : November 13, 2026 at 12:32 PM UTC-8:00 (PST)
  Experiment       : nsd-mini
  Subject          : sdbp_nsdmini_0002
  Runs             : 10
  MEG/EEG data     : None
  MRI data         : 48 scans
  Checkpoint       : within motion correction
  Checkpoint saved : November 20, 2026 at 9:29 PM PST
  Saved by         : justin


---

Display information about a session


---

In [ ]:
# display the top-level information
session.print()

In [ ]:
# more detailed trial-by-trial information, for a particular run
session.print(runNum=0, verbose=True)

In [ ]:
# display mri scans
session.mri.print()

In [ ]:
# display behavioral plots
session.display();

---

Show history of actions

---

In [ ]:
session.history()

In [ ]:
# show more detailed information
session.history(verbose=True)

In [ ]:
# show what products actions produced
session.history(showProducts=True)

In [ ]:
# show the logs for a particular action
session.history(3,showLogs=True)

In [ ]:
# get the products for a particular action and print
motion = session.getProduct("motion")
motion.print()

In [ ]:
# dispaly any plots related to the product
motion.display();

---

AFNI: check availability and optional install

---

In [ ]:
# Install AFNI. If already installed, will not try to re-install but will display information about installation
afni = pglAFNI()
afni.install()

In [ ]:
# check availability of specifc commands
afni = pglAFNI(directory="~/proj/afni/abin")
available = afni.checkAvailable("3dvolreg")

---

Run motion comp

---

In [ ]:
newSession = pglActionsMRI.mriEstimateMotionAFNI().execute(session=session, deferApply=True)
if newSession: session = newSession

In [ ]:
# display matrices
motion = session.getProduct('motion')
if motion: motion.display()

---

Reconstruct volume triggers fix (only needed if broken)

---

In [ ]:
newSession = pglActionsFixes.reconstructVolumeTriggersFromNifti.execute(session=session)
if newSession: session = newSession

---

GLMsingle

---

In [ ]:
pglGLMsingle.configure()

In [ ]:
newSession = pglActionsMRI.GLMsingle.execute(session=session, model="B", stimDur=2.0, outputName="GLMsingleTypeBTest")
if newSession: session = newSession

---

Export session to another database

---

In [4]:
session.export('lagavulin')

             0% [   00:00<       ?] Exporting saved session

(pglDatabase:exportSession) Exporting sessionID=136 from local (/Users/justin/data/pgl) to lagavulin (/Users/Shared/data/pgl)
Local changes are not included; the working session remains bound to its source.


error ignored in rollback of <psycopg.Transaction (rolled_back_with_error) [BAD] at 0x16c01cc80>: consuming input failed: server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.


❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(pglDatabase:_warnSessionExportFailure) SSHException: Server connection dropped:

Session export stopped during preparingPayloads.
The final destination commit was not attempted.
Payload preparation started. Created or partial output is retained; this does not establish that preparation completed.
Transaction/resource cleanup was attempted as contexts unwound; successful cleanup is not asserted by this warning.
Source checkpoint: 136 in schema UUID 45391270-8d26-42d8-8b04-ae1e575a2cd9.
Destination schema UUID: 09c1021a-e887-40be-9316-aa8664f128a0.
Destination backend ID: 1.
Allocated preparation prefix: export-59bf154a-f8d5-4eb5-93dd-66c3d48deb0a (relative to the configured backend root;
allocation alone does not establish that files were created).
No automatic retry, connection termination, or payload deletion occurred.

What to do next:
Resolve the error above before another attempt. Review the original 

SSHException: Server connection dropped: 

In [4]:
session.export('lagavulin')

             0% [   00:00<       ?] Exporting saved session

(pglDatabase:exportSession) Exporting saved checkpoint 136 to profile 'lagavulin'. Unsaved working edits are not included; the working session
remains bound to its source.


Socket exception: Can't assign requested address (49)
error ignored in rollback of <psycopg.Transaction (rolled_back_with_error) [BAD] at 0x31dd28b60>: consuming input failed: server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.


❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(pglDatabase:_warnSessionExportFailure) SSHException: Server connection dropped:

Session export stopped during preparingPayloads.
The final destination commit was not attempted.
Payload preparation started. Created or partial output is retained; this does not establish that preparation completed.
Transaction/resource cleanup was attempted as contexts unwound; successful cleanup is not asserted by this warning.
Source checkpoint: 136 in schema UUID 45391270-8d26-42d8-8b04-ae1e575a2cd9.
Destination schema UUID: 09c1021a-e887-40be-9316-aa8664f128a0.
Destination backend ID: 0.
Allocated preparation prefix: export-40f7700f-49f6-4714-9822-f439b8f99c05 (relative to the configured backend root;
allocation alone does not establish that files were created).
No automatic retry, connection termination, or payload deletion occurred.

What to do next:
Resolve the error above before another attempt. Review the original 

SSHException: Server connection dropped: 

In [ ]:
session.saveSession(checkpointName="within motion correction")

In [ ]:
from pgl.pglExternalTools import pglExternalTool, pglAFNI

pglAFNI.configure()

In [ ]:
from pgl import pglExternalTool
reports = pglExternalTool.print()

In [ ]:
from pgl.pglExternalTools import pglMATLAB

reports = pglMATLAB.print(validate=True, timeout=120)

In [ ]:
#session.setScanRunMatches([(4, 0)])
session.displayScanRunMapping()

In [ ]:
session.history()

In [ ]:
session.history(showProducts=True)
product = session.getProduct(4)

In [ ]:
product.display()

In [ ]:
from pgl.pglActionsMRI import pglActionsMRI
from pgl.pglDatabase import pglDatabase

session = pglDatabase.loadSession(136)
session.setScanRunMatches([(4, 0)])

session = pglActionsMRI.GLMsingle.execute(session=session,model="B", scanIndices=[4], stimDur=2.0, outputName="GLMsingleTypeBTest")